# Market Base
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/narratech/market-base/blob/main/notebooks/market_base.ipynb)

| ***Campo*** | ***Detalle*** |
| :--- | :--- |
| **Asignatura** | Aprendizaje automático y minería de datos |
| **Profesor** | Federico Peinado |
| **Autores** | Pablo Iglesias Rodrigo [@Paigro](https://github.com/Paigro), Nieves Alonso Gilsanz [@nievesag](https://github.com/nievesag)  |
| **Institución** | Universidad Complutense de Madrid |
| **Licencia** | Copyright © 2026 Federico Peinado. Distribuido bajo licencia [MIT](https://opensource.org/licenses/MIT) |
| **Requisitos** | Python 3.x (Módulos nativos, sin dependencias externas) | 

Este punto de partida está pensado para prácticas sobre Minería de Datos. Antes de alimentar cualquier modelo analítico o de aprendizaje automático, es imprescindible adquirir, limpiar, explorar y visualizar a fondo los datos. 

### Características del estudio
A. Adquisición del conjunto de datos y exploración inicial
B. Valoración de la calidad de datos, limpieza e ingeniería de características
C. Análisis de variables
D. Análisis de correlaciones y serie temporal
E. Nichos de oportunidad y conclusiones

## 🛠️ Configuración Inicial e Importación de Librerías

In [ ]:
import os # Para interactuar con el sistema operativo
import urllib.request  # Para descargar archivos desde la web
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo gráfico
sns.set_theme(style="whitegrid") # Un estilo más limpio y profesional para los gráficos
plt.rcParams['figure.figsize'] = (10, 6) # Para evitar que los gráficos se vean demasiado pequeños

print("Bibliotecas cargadas correctamente.")

## A. Adquisición del conjunto de datos y exploración inicial

**Objetivo:** Descargar el conjunto de datos directamente desde el repositorio del proyecto en la carpeta local `data/raw/` y verificar que la estructura de columnas y tipos de datos se haya cargado correctamente sin desfases en la cabecera.

In [ ]:
# 1. Rutas locales y URL oficial
PROJECT_ROOT = os.path.abspath(os.path.join(".."))
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
LOCAL_CSV_PATH = os.path.join(RAW_DATA_DIR, "games.csv")
DATASET_URL = "https://github.com/narratech/market-base/releases/download/Release/games.csv"
import csv

# 2. Descarga automatizada a data/raw/
if not os.path.exists(LOCAL_CSV_PATH):
    os.makedirs(RAW_DATA_DIR, exist_ok=True)
    print("Descargando games.csv desde el repositorio...")
    urllib.request.urlretrieve(DATASET_URL, LOCAL_CSV_PATH)
    print(f"¡Descarga completada en {LOCAL_CSV_PATH}!")
else:
    print(f"El archivo ya existe en {LOCAL_CSV_PATH}.")


#encabezado = "AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies"
#with open (LOCAL_CSV_PATH, "r", encoding="utf-8") as f:
#    lineas = f.readlines()
#    for linea in lineas[1:]: # 1: se salta la primera linea para ignorar el encabezado
#        linea_limpia = linea.strip() # quita los espacios
#        columnas = linea_limpia.split(", ") # separa los datos

#df = pd.read_csv(LOCAL_CSV_PATH,
#    encoding="utf-8", 
#    on_bad_lines="skip",  # Para que no coja las lineas malas.
#    engine='python') # El motor de Python para Pandas es mas permisivo que el otro con las cosas y as ino da error.


#print(df.info())

#  En clase hemos visto:
with open(LOCAL_CSV_PATH, "r", encoding="utf-8") as f:
    encabezado = f.readline().strip() # Primera linea, encabezado. Strip para quitar espacios en blanco al principio y final.

encabezado_corregido = encabezado.replace("DiscountDLC count", "Discount,DLC count") # El archivo original no tenia coma entre ambas columnas. Lo cambiamos a mano.
nombres = encabezado_corregido.split(",") # Encabezado corregido y separado por comas.

# Para cargar.
df_raw = pd.read_csv(LOCAL_CSV_PATH,
                     skiprows=1, # Como la primera linea es el encabezado y no son ddatos la quitamos.
                     names=nombres, # Como no hemos cogido el encabezado entonces los ponemos a mano.
                     encoding="utf-8") # Codigicacion del archivo.

print("[M] Data frame creado.")


In [ ]:
# TODO: Muestra las dimensiones del DataFrame, sus primeras filas y el resumen de información técnica (dtypes, valores no nulos)
# Escribe tu código aquí:

print("[M]----------Dimensiones----------")
print(df_raw.shape)

print("\n")

print("[M]----------Primeras filas----------")
print(df_raw.head(3))

print("\n")

print("[M]----------Resumen----------")
print(df_raw.info())

print("\n")

print("[M]----------Datos ausentes----------")
print(df_raw.isnull().sum())

> **Discusión**  
> *Redacta aquí tus observaciones sobre la estructura de los datos, número de registros, alineación de cabeceras o columnas con tipos de datos no optimizados.*

El CSV tiene un error en la cabecera, dos columnas no estan separadas por una copia. Estan "DiscountDLC count" en vez de "Discount, DLC count".

---

## B. Valoración de la calidad de datos, limpieza e ingeniería de características

**Objetivo:** Tratar valores nulos, transformar tipos de datos (como la conversión de cadenas de texto separadas por comas a listas iterables) y calcular nuevas métricas como la tasa de valoraciones positivas, aplicando un filtrado adecuado para evitar sesgos por bajo volumen de reseñas.

In [ ]:
# TODO: Tratamiento de nulos y conversión de tipos de datos (ej. cadenas a listas para géneros/tags)
# Escribe tu código aquí:

# Creamos una copia del dataframe por si acaso y por buena practica. Modificamos la copia en vez del original.
df=df_raw.copy()

# Quitamos columnas que por ejemplo tienen muchos nulos o que no nos interesen para 
columnasEliminar=["Movies", "Screenshots", "Notes", "Score rank"]
# Quitar columnas que no interesan.
df=df.drop(columns=columnasEliminar) # o con "df.drop(columns=columnasEliminar, inplace=true)"

# Convertir las fechas de tipo str a formato Date de Pandas.
df["Release date"] = pd.to_datetime(df["Release date"], # El que.
                                    format="mixed", # Para que pueda haber fechas en varios formatos. Pandas los sabe interpretar la mayoria.
                                    errors="coerce") # Convierte las fechas valiadas a NaT (Not a Time)

# Valores de texto que pueden estar nulos.
columnasTexto = ["Developers", "Publishers", "Genres", "Categories", "Tags"]
etiqueta="UnknowName"
for col in columnasTexto:
    df[col] = df[col].fillna(etiqueta) # Llena los valores ausentes con la etiqueta que queramos.

# Transformaciones.
def cadenaALista(cadena):
    if isinstance(cadena, str) and cadena != etiqueta: # Comprobamos que la cadena sea str y que no tenga la etiqueta que le hemos puesto antes.
        return[item.strip() for item in cadena.split(",")] # Separamos las cadenas qeu tengan varios str separandolos habeindolos previamente separado por comas.
    else:
        return []

variasCadenas=["Genres", "Categories", "Tags"] # Las columnas que queramos separar.
for col in variasCadenas:
    df[col]=df[col].apply(cadenaALista) # Que aplique nuetra funcion

#TODO faltaria lo de crear nueva columna de tasa de valoreacines positivas (positivas/total reseñas)*100 para los juegos que tengan mas de x reseñas



In [ ]:
# TODO: Crear la característica 'tasa_positividad' filtrando solo juegos con más de 50 reseñas totales
# Escribe tu código aquí:

nReviews = df["Positive"] + df["Negative"]

minReviews = 50
nonEnought = -1 # Valor a poner si no hay suficientes valoraciones.

# 3. Cálculo de la tasa individual para cada juego
df["tasa_positividad"] = np.where(nReviews > minReviews, # Solo si se cumple que tenga min 50 valoraciones.
                                (df["Positive"] / nReviews) * 100, # El valor que calculado.
                                nonEnought # Sino lo llenamos con nuestro valor.
                                ).round(2) # Para que solo tenga 2 decimales.

print(df[["Name", "Positive", "Negative", "tasa_positividad"]].head(50)) # Para comprobar.

> **Discusión**  
> *Explica el impacto del tratamiento de nulos y justifica cuantitativamente por qué es técnicamente necesario filtrar por un volumen mínimo de reseñas (50) para evitar distorsiones en la métrica de positividad.*

---

## C. Análisis de variables

**Objetivo:** Analizar los géneros más frecuentes en oferta frente a su volumen estimado de jugadores. Analizar estadísticamente la distribución de precios en la plataforma.

In [ ]:
# TODO: Desempaquetado de géneros y cálculo de frecuencias absolutas y relativas.
# Ej. Generar gráfico de barras del Top 10 de géneros.
# Escribe tu código aquí:


In [ ]:
# TODO: Separar juegos de pago de los gratuitos. Calcular estadísticos descriptivos (media, mediana, IQR). 
# Ej. Generar un histograma/boxplot de la distribución de precios.
# Escribe tu código aquí:


> **Discusión**  
> *Comenta qué género domina la oferta y si coincide con los más jugados. Explica la diferencia entre la media y la mediana en el precio para identificar el rango habitual de precios en Steam.*

---

## D. Análisis de correlaciones y serie temporal

**Objetivo:** Estudiar la relación cruzada entre el precio y el éxito del videojuego. Analizar la serie temporal de lanzamientos de juegos por año en Steam.

In [ ]:
# TODO: Calcular la matriz de correlación (ej. Spearman) entre precio, positividad y número de reseñas.
# Ej. Generar visualización (Heatmap o Scatter plot con tendencia).
# Escribe tu código aquí:


In [ ]:
# TODO: Extraer el año de lanzamiento de 'Release date' y analizar la evolución del catálogo de Steam año a año.
# Ej. Generar gráfico de línea/área.
# Escribe tu código aquí:


> **Discusión**  
> *¿Tienen los juegos más caros mejor valoración? Analiza los resultados de correlación y describe el ritmo de crecimiento anual de la plataforma.*

---

## E. Nichos de oportunidad y conclusiones

**Objetivo:** Identificar combinaciones o nichos de mercado con baja saturación de oferta pero alta tasa de positividad media (u otro análisis avanzado que queráis plantear) y sintetizar las conclusiones generales del estudio.

In [ ]:
# TODO: Código para cruzar géneros/tags poco saturados con altas valoraciones o análisis avanzado a elección.
# Escribe tu código aquí:


> **✏️ Conclusiones**  
> *Redacta un breve informe de cierre sintetizando los hallazgos clave obtenidos a lo largo de todo el estudio. Propón recomendaciones concretas basadas en datos para un nuevo estudio de desarrollo que quiera lanzar un juego en Steam.*